# L15a: Testing a Trading Strategy
Today, we'll introduce a trading method that updates its estimates of each stock every trading day and picks a new basket of stocks when the market changes. Then we'll test strategies built on this method, first with a backtest on the real market and then with a stress test on synthetic markets.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Update the estimates of a trading method:__ Update the single index model of each stock every trading day with exponentially weighted least squares. Use the updated estimates in the preference weights and the market hedge of the trading method.
> * __Backtest a strategy on the real market:__ Describe the daily loop of a backtest and the six triggers that start a new decision. Compare a strategy from the menu with `SPY` and with the other strategies on the menu.
> * __Stress-test a strategy with the five gates:__ Run a strategy on many synthetic markets. Test its median outcome and its share of losing markets against the five gates.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * Exponentially weighted least squares updates the single index model of each stock with every new trading day, and the weight of a day halves every 63 trading days. The preference weights and the market hedge then follow each stock's recent market sensitivity.
> * A backtest runs a strategy on the real market one trading day at a time. Each day, the strategy uses only the data through that day, and it makes a new decision when one of six triggers fires.
> * A stress test runs a strategy on many synthetic markets. Four of the five gates take the median over the markets, and the failure rate counts the markets that end below the starting wealth.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * The running sums of the daily update, and the update of the general factor model.
> * The settings of the six triggers, such as the 8% drawdown.
> * How the practicum's engine differs from the trading method.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L15a) lists this lecture's symbols with their meanings and units.

Let's get started!

___

## Examples

__We will work through these examples in lecture:__

> [▶ Backtest a strategy from the menu](CHEME-5660-L15a-Example-Strategy-Backtest-Fall-2026.ipynb). In this example, we backtest one strategy from the menu on the real market, from January 2, 2025, to November 24, 2026. We mark the days a trigger fired, place the strategy among the saved backtests of the whole menu, and run it again with updated estimates.

> [▶ Stress-test a strategy with the five gates](CHEME-5660-L15a-Example-Strategy-Stress-Test-Fall-2026.ipynb). In this example, we stress-test one strategy from the menu on 200 synthetic markets and plot the range of its wealth. We test the strategy with the five gates, and then test it again with the market hedge on.

__Review on your own:__

> [▶ Update the single index model and the preference weights](CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). In this example, we update the single index model of every stock with each trading day of 2025 and 2026, starting from the 2016 to 2024 fit. We compute the preference weights from the updated estimates and pick the preference basket once a month and whenever the market turns. Then we compare the preference basket with the basket from the fixed 2016 to 2024 estimates.

___

## Updating the Single Index Model
Previously, we fit the single index model of each stock once, on the training data, and held its estimates fixed. However, a stock's market sensitivity changes over time. By late 2026, most stocks in our universe moved less with the market than their 2016 to 2024 estimates say. We keep the estimates current by updating them with each new trading day, giving recent days the most weight:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-EWLS-JPM.svg" width="740" style="max-width:100%; height:auto;" alt="The market sensitivity of JPM over 2025 and 2026. A dashed line marks the fixed estimate from the 2016 to 2024 fit, 1.12. The blue updated estimate, with one update per trading day, starts at 1.12, stays near 1.0 through 2025, and drops to about 0.7 in mid-2026. Late in 2026, JPM moves less with the market than the fixed estimate says."/>
    </center>
</div>

Previously, we estimated $\alpha_{i}$ and $\beta_{i}$ by least squares, where we minimized the sum of squared residuals with every training day counted the same. However, we could include __new information__ as it arrives by weighting the new days more than the old ones. Let's consider exponentially weighted least squares (EWLS):

> __Exponentially weighted least squares (EWLS):__ On day $t$, the updated baseline $\alpha_{i}(t)$ and market sensitivity $\beta_{i}(t)$ of stock $i$ are the solutions to the weighted least-squares problem:
>
> $$
> \left(\alpha_{i}(t),\beta_{i}(t)\right) = \arg\min_{\alpha_{i},\,\beta_{i}}\Biggl[\underbrace{\sum_{k=1}^{t}\phi^{\,t-k}\,\bigl(\overbrace{g_{i}(k)-\alpha_{i}-\beta_{i}\,g_{m}(k)}^{\text{residual }r_{i}(k)}\bigr)^{2}}_{\text{new days}} + \underbrace{\phi^{\,t}\,\frac{N_{0}}{2263}\sum_{k=-2262}^{0}r_{i}(k)^{2}}_{\text{prior}}\Biggr]
> $$
>
> where $g_{i}(k)$ and $g_{m}(k)$ are the daily growth rates of stock $i$ and `SPY` on day $k$ (1/yr). Day $0$ is December 31, 2024, the last day of the 2016 to 2024 fit, day $1$ is January 2, 2025, and day $476$ is November 24, 2026, the last day of our data.
>
> The problem has three parts:
>
> * __New days:__ The weight of day $k$ is the decay $\phi=2^{-1/63}\approx 0.989$ raised to the age of the day, $t-k$ trading days. For example, on day $3$, January 6, 2025, the first sum is $\phi^{2}\,r_{i}(1)^{2}+\phi\,r_{i}(2)^{2}+r_{i}(3)^{2}$. The weight of a day halves every 63 trading days, about three months.
> * __Prior:__ Each of the 2,263 days of 2016 to 2024 gets the weight $N_{0}/2263$, with $N_{0}=63$. On day $0$, the prior alone gives the 2016 to 2024 fit. After that, the factor $\phi^{\,t}$ shrinks the prior's day $0$ weight.
> * __No look-ahead:__ The estimates on day $t$ use the growth rates through day $t$ and nothing later. The trading method decides after the close of day $t$ and trades at the next open. However, this is still a backward-looking estimate.

Details of the implementation, which updates the estimates from a few running sums, are shown in [the EWLS recursion notebook](advanced/ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb).

### Updating the General Factor Model
Previously, we fit the general factor model once, with every training day counted the same. The same exponential weights can update it with each new trading day:

* __Market, economic, and sector loadings:__ Steps 1 and 3 of the fitting algorithm become weighted least-squares fits. On day $t$, they use the weights of the EWLS problem, including the prior from the 2016 to 2024 fit.
* __Sector factors:__ Step 2 computes the sector factors of day $k$ from the residuals of that day alone. It needs no weights.
* __Factor averages and covariance:__ The averages $\boldsymbol{\mu}$ and the covariance matrix $\mathbf{\Sigma}_{Y}$ of the market, economic, and sector factors become exponentially weighted, with the same weights, before the Cholesky step.

Refitting the jump hidden Markov models is the slow part of an update. They can be refit less often than the loadings.

### Preference Weights with Updated Estimates
Previously, we computed the preference weight $\gamma_{i}(t)$ of each stock and used it to rank the stocks for inclusion in the portfolio. The preference weight depends on the market state $\xi_{t}$, which compares a 21-day and a 63-day EMA of the `SPY` price:

$$
\xi_{t} = -G\left(\frac{\bar{S}^{\text{short}}_{t}}{\bar{S}^{\text{long}}_{t}}-1\right)
$$

where the gain $G>0$ scales $\xi_{t}$ so that $|\xi_{t}|\leq 1$ on 99% of training days. When the market falls, recent prices sit below the quarter's average, and $\xi_{t}$ turns positive. The day $\xi_{t}$ changes sign is a market turn, the regime change that the trading method acts on.

The estimates and the preference weights update every trading day, but the basket of stocks we hold changes only on pick days:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Gamma-Picks.svg" width="860" style="max-width:100%; height:auto;" alt="One time axis with three rows. The top row is a path of the market state xi t, shaded blue where the market is rising and red where it is falling. The middle row is a line with a dot for every trading day, when alpha, beta, and gamma update. The bottom row shows the preference baskets held, numbered 1 to 7, each from one pick day to the next. Most baskets last 21 trading days. A brace over basket 2 reads: 15 trading days, cut short by the market turn. Where xi t crosses zero, an orange dashed line marks a market turn: the method picks a new basket, and the 21-day count restarts."/>
    </center>
</div>

The trading method of [Varner (2026)](https://arxiv.org/abs/2611.03022) uses the same preference weight to choose its stocks:

> __Preference model (updated estimates):__ For stock $i$ on day $t$, the preference weight is given by:
>
> $$
> \boxed{\begin{aligned}
> \gamma_{i}(t) &= \tanh\left(h\,\frac{\tilde{g}_{i,t}}{\max\left(|\beta_{i}(t)|,\,\beta_{\text{floor}}\right)^{\xi_{t}}}\right)\\
> &= \tanh\left(h\,\frac{\alpha_{i}(t)+\beta_{i}(t)\,\tilde{g}_{M,t}}{\max\left(|\beta_{i}(t)|,\,\beta_{\text{floor}}\right)^{\xi_{t}}}\right)
> \end{aligned}}
> $$
>
> where $\tilde{g}_{i,t}$ is the expected growth rate of stock $i$ in today's market (1/yr), the horizon $h=1$ year makes the argument of the $\tanh$ dimensionless, and the floor is $\beta_{\text{floor}}=0.1$. With a fixed $\beta_{i}\geq\beta_{\text{floor}}$, this is the preference weight we used before.
>
> The second line computes $\tilde{g}_{i,t}$ with the single index model, where the recent market growth rate $\tilde{g}_{M,t}$ (1/yr) is a 252-day EMA of the market growth rates. The single index model is one choice. A different factor model, or the stock's past growth rates alone, could give $\tilde{g}_{i,t}$ instead.
>
> The trading method makes three changes:
>
> * __Updated estimates:__ The estimates $\alpha_{i}(t)$ and $\beta_{i}(t)$ come from the daily EWLS update. The preference weight follows the changes in each stock's market sensitivity. AQR's [betting-against-beta portfolio](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta) also re-estimates the market sensitivity of each stock from a rolling window of past returns.
> * __A floor on the market sensitivity:__ Some stocks have an updated market sensitivity near or below zero. The denominator uses $|\beta_{i}(t)|$ with a floor, which keeps the denominator positive for every stock. The sign of the preference weight is then the sign of the expected growth rate $\tilde{g}_{i,t}$.
> * __Picks when a trigger fires:__ The __preference basket__ holds the stocks with the highest preference weights in each sector. We choose how many stocks to keep from each sector. The method picks a new basket whenever one of six triggers fires, for example after 21 trading days or at a market turn.

### The Market Hedge
The trading method can also run with a __market hedge__, which limits the market sensitivity of the whole portfolio. The limit drops as the market falls:

<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Market-Hedge.svg" width="720" style="max-width:100%; height:auto;" alt="A line shows the limit on the market sensitivity of the portfolio, beta P of t, against the market state xi t. The limit falls from 2 at xi t equals minus 1 to 1 at xi t equals 0, the market sensitivity of SPY. It reaches 0 at xi t equals 1, the market sensitivity of betting against beta. Below zero, a red area marks where the method sells SPY short, and the line flattens at a floor of minus 0.5. Arrows from the line point down: beta P of t stays on or below the line. A gray band marks where the absolute value of xi t is at most 1, which holds on 99 percent of training days."/>
    </center>
</div>

> __Market hedge:__ On each day $t$ that the trading method sets its holdings, the hedge requires:
>
> $$
> \boxed{\beta_{P}(t) = \sum_{i}w_{i}(t)\,\beta_{i}(t) \leq \max\left(1-\xi_{t},\,-0.5\right)}
> $$
>
> where $\beta_{P}(t)$ is the market sensitivity of the portfolio, and $w_{i}(t)$ is the fraction of the portfolio's value in holding $i$, negative for a short. Cash has a market sensitivity of zero. Let's consider a few cases:
>
> * __Market rise:__ When the market rises ($\xi_{t}<0$), the limit on $\beta_{P}(t)$ is above one, so the hedge allows more market risk than `SPY`. Because the method never borrows to buy stocks, $\beta_{P}(t)$ rises above one only when the stocks it holds move more than the market on average.
> * __Flat or falling market:__ In a flat market ($\xi_{t}=0$), the limit on $\beta_{P}(t)$ is one, the market sensitivity of `SPY`. As the market falls, the limit drops to zero at $\xi_{t}=1$. To keep $\beta_{P}(t)$ at or below the limit, the method can move money into the basket's stocks with a low market sensitivity, hold cash, or buy up to five stocks whose market sensitivity is zero or negative.
> * __Steep fall:__ After a steep fall ($\xi_{t}>1$), the limit is below zero. The method then holds stocks and cash that are uncorrelated or anti-correlated with the market, and sells `SPY` short to push $\beta_{P}(t)$ below zero. Because `SPY` has a market sensitivity of one, a short worth 20% of the portfolio's value lowers $\beta_{P}(t)$ by $0.2$.
>
> The floor of $-0.5$ on the limit caps the `SPY` short at half of the portfolio's value.


Let's update the estimates of every stock in our universe and pick the preference basket:

> __Review on your own:__
>
> [▶ Update the single index model and the preference weights](CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb). In this example, we update the single index model of every stock with each trading day of 2025 and 2026, starting from the 2016 to 2024 fit. We pick the preference basket once a month and whenever the market turns, and compare it with the basket from the fixed estimates.

The example ends with the preference basket of the last pick day in our data.

__How does the trading method compare with betting against beta?__ Previously, we met [AQR Capital Management](https://www.aqr.com/Our-Firm/About-Us), where Andrea Frazzini and Lasse H. Pedersen proposed the [betting-against-beta portfolio](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta) in 2014. Their portfolio always buys stocks with a low market sensitivity and sells stocks with a high one, in amounts that make its market sensitivity zero. The trading method makes the same bet only when the market falls:

* __Signal__: When the market falls ($\xi_{t}>0$), the denominator is above one for a stock with $|\beta_{i}(t)|>1$ and below one for a stock with $|\beta_{i}(t)|<1$. Among stocks with a positive expected growth rate, the preference weight then favors stocks that move less with the market. When the market rises ($\xi_{t}<0$), the tilt reverses.
* __Long-only basket__: The preference basket keeps the stocks with the highest preference weights in each sector. A stock with a low preference weight is left out, not sold short.
* __Market hedge__: With the hedge on, the limit on $\beta_{P}(t)$ reaches zero, the market sensitivity of betting against beta, at $\xi_{t}=1$. Betting against beta reaches zero by selling stocks with a high market sensitivity. The trading method sells only `SPY` short, and only after a steeper fall takes the limit below zero.

___

## Backtesting a Strategy
The trading method leaves four choices to us. We call one set of these choices a __strategy__:

| Choice | What it sets | On the [strategy menu](strategies/CHEME-5660-L15a-Strategy-Menu-Fall-2026.ipynb) |
|:--|:--|:--|
| Picking rule | which tickers go into the basket | 9 entries, for example the highest preference weights in each sector, or a random draw |
| Weighting rule | how the wealth is split among the stocks of the basket | 9 rules, for example Cobb–Douglas, minimum variance, or equal weight |
| Market hedge | whether the market sensitivity of the portfolio has a limit | on or off |
| Estimates | which estimates of the single index model the method uses | fixed from the fit on the training data, or updated every trading day |


We test a strategy first on the real market. Each trading day of the test follows the same steps:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Backtest-Day.svg" width="860" style="max-width:100%; height:auto;" alt="One time axis with three trading days, day t minus 1, day t, and day t plus 1, each drawn as a bar from its open to its close. A dashed gray arc from the close of day t minus 1 to the open of day t reads: fill yesterday's orders. An arrow rises from the close of day t to a box labeled after the close: compute the wealth W of t, update the estimates, and check the six triggers. A gray arrow to the left of the box reads: no trigger, keep the portfolio. An orange arrow labeled a trigger fires rises to an orange box, make a new decision, with three steps. Step 1 picks a basket with the picking rule. Step 2 weights it with the weighting rule within the trading limits, and within the market sensitivity limit if the hedge is on. Step 3 queues the orders. An arrow from step 3 runs down to the open of day t plus 1, labeled orders fill at the open."/>
    </center>
</div>

> __Definition (backtest):__ A backtest runs a strategy on past market data, one trading day at a time, as if we had traded it. On each day, the strategy uses only the data through that day, and its orders fill at the next day's opening prices.

### Backtest Algorithm
__Initialize__: Given a strategy; the prices, volumes, and Treasury bill rates of the training and backtest days; the estimates of the single index model of each stock on the day before the backtest starts; and the starting wealth $W_{0}$, all in cash. The first decision comes after the close of the first day.

For each trading day $t$ of the backtest __do__:

1. __Fill:__ Fill the orders queued yesterday at today's opening prices. Each trade pays half the bid-ask spread and a market impact that grows with the size of the trade.
2. __Value:__ Grow the cash at the Treasury bill rate. Compute today's wealth $W(t)$ at the closing prices: the cash plus the value of every holding.
3. __Update:__ If the strategy uses updated estimates, __then__ update the single index model of each stock with today's growth rates. Compute the market state $\xi_{t}$ and the preference weight $\gamma_{i}(t)$ of each stock.
4. __Decide:__ If no trigger fires, __then__ keep the portfolio. Otherwise, make a new decision:
   - a. __Pick:__ The picking rule chooses the stocks of the basket.
   - b. __Weight:__ The weighting rule sets the weight of each stock within the trading limits: a limit on the weight of each ticker, a volatility no higher than that of `SPY` on the training data, a limit on the trading costs of each decision, and no borrowing. With the hedge on, the market sensitivity of the portfolio must also stay at or below its limit.
   - c. __Queue:__ Turn the weights into share counts at today's closing prices, queue the orders for tomorrow's open, and forecast the wealth over the next 21 trading days with the __generator__, the general factor model of the practicum.

__Output__: The wealth $W(t)$ of each day, the decisions with the trigger that started each one, and the trades.

### The Six Triggers
After each close, the trading method checks six triggers:

| Trigger | Fires when |
|:--|:--|
| Scheduled | 21 trading days have passed since the last decision. |
| Market turn | The market state $\xi_{t}$ has changed sign since the last decision. With the hedge on, it also fires when $\xi_{t}$ crosses one, where the `SPY` short starts or stops. |
| Band exit | The wealth leaves the middle 95% of its forecast. |
| Drawdown | The wealth falls more than 8% below its peak. The peak starts over at each drawdown decision. |
| Market sensitivity over its limit | With the hedge on, the market sensitivity $\beta_{P}(t)$ of the portfolio is more than 0.1 above its limit. |
| Back from cash | The scheduled trigger, when the last decision put all the wealth in cash. |

Let's backtest one strategy from the menu:

> __Example:__
>
> [▶ Backtest a strategy from the menu](CHEME-5660-L15a-Example-Strategy-Backtest-Fall-2026.ipynb). In this example, we backtest one strategy from the menu on the real market, from January 2, 2025, to November 24, 2026. We mark the days a trigger fired, place the strategy among the saved backtests of the whole menu, and run it again with updated estimates.

The backtest shows how the strategy did on the one market that happened.

___

## Stress-Testing a Strategy
Next, we run the same strategy on many synthetic markets from the generator, to see the range of outcomes it could face over the next year:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L15a-Stress-Test.svg" width="860" style="max-width:100%; height:auto;" alt="Thin red lines fan out from one point on a vertical dashed line labeled start day. The point is the starting wealth W 0. Each line is the wealth path on one synthetic market from the generator, over one synthetic year of 252 trading days, and the fan widens over the year. At the right edge, a sideways histogram shows the wealth on the last day over the 200 synthetic markets. A dashed horizontal line marks the starting wealth W 0, and a solid tick above it marks the median. The bars below W 0 are a darker red, labeled failures: end below W 0. The lines that end below W 0 have the same darker red, and the other lines and bars are a light red."/>
    </center>
</div>

> __Definition (stress test):__ A stress test runs a strategy on $P$ synthetic markets. Each market is the real history to the start day, the last day of our data, followed by one synthetic year of 252 trading days drawn from the generator. On every market, the strategy starts with the wealth $W_{0}$ in cash and with its estimates on the start day.

### Stress-Test Algorithm
__Initialize__: Given a strategy; the real history to the start day; the generator; and the number of markets $P=200$.

For each market $p=1,2,\dots,P$ __do__:

1. __Draw:__ Draw the growth rates of `SPY` and of every ticker for 252 trading days from the generator. Compound each price from its last real closing price.
2. __Backtest:__ Run the backtest algorithm from the start day through the 252 synthetic days.
3. __Score:__ Compute the Sharpe ratio, the largest drawdown, the wealth on the last day, and the net present value of the run.

__Output__: The scores of the strategy on each of the $P$ markets.

### The Five Gates
We test the strategy's scores against five gates:

| Gate | Passes when | What it measures |
|:--|:--|:--|
| Median Sharpe ratio | at least 0.3 | the growth of the wealth over the Treasury bill rate, per unit of volatility |
| Median largest drawdown | at most 25% | the largest fall of the wealth below its peak |
| Failure rate | at most 10% | the share of markets on which the wealth ends below $W_{0}$ |
| Median wealth on the last day | at least the comparison's median | the wealth after one year |
| Median net present value | at least USD 0 | the wealth on the last day, discounted at the Treasury bill rate, minus $W_{0}$ |

In the fourth gate, a strategy with fixed estimates faces the 60/40 benchmark, which holds 60% of its wealth in `SPY` and 40% in the long-term Treasury bond fund `TLT`. A strategy with updated estimates faces its copy with fixed estimates on the same markets, which tests whether updating the estimates added wealth.

Let's stress-test one strategy from the menu:

> __Example:__
>
> [▶ Stress-test a strategy with the five gates](CHEME-5660-L15a-Example-Strategy-Stress-Test-Fall-2026.ipynb). In this example, we stress-test one strategy from the menu on 200 synthetic markets and plot the range of its wealth. We test the strategy with the five gates, and then test it again with the market hedge on.

___

## Testing Your Fund in the Practicum
The practicum uses the same data, the same Treasury bill rate, the same 200 synthetic markets, and the same five gates as our examples. Its engine differs from the trading method in five ways:

| | The trading method | The practicum's engine |
|:--|:--|:--|
| __Triggers__ | the six triggers | two of the six: scheduled and market turn |
| __Weights__ | the weighting rule's formula, adjusted to meet the trading limits | your formula, as you write it |
| __Fills__ | at the next day's opening price | at the closing price of the decision day |
| __Limits__ | the trading limits of the backtest | the leverage you set, from 1.0 to 2.0, with margin interest and a maintenance rule |
| __Hedge__ | the market hedge, on or off | no market hedge, but protective puts can limit the fund's losses |

The same picking and weighting rules can score differently in the practicum. You can send a fund that fails a gate to the paper test if you explain why.

___

## Optional Advanced Material
> [▶ Derive the daily update of the single index model](advanced/ewls/CHEME-5660-L15a-Derivation-EWLS-Update-Fall-2026.ipynb). In this notebook, we derive the running sums that the exponentially weighted least squares update carries from one day to the next. We compute the updated estimates and the residual standard deviation from these sums, and show how the 2016 to 2024 fit sets the starting values of the sums.

___

## Summary
We updated the estimates of the trading method every trading day, backtested strategies on the real market, and stress-tested them on synthetic markets with the five gates.

> __Key Takeaways:__
>
> * __Updated estimates:__ We updated the single index model of each stock every trading day with exponentially weighted least squares, starting from the fixed fit. The preference weights and the market hedge used the updated estimates. Across the menu, updating the estimates added wealth in fewer than half of the backtests.
> * __Backtests and triggers:__ We ran strategies on the real market one trading day at a time, and each decision used only the data through that day. A new decision started when one of six triggers fired, such as 21 trading days passing or a market turn. Most strategies on the menu had a lower Sharpe ratio than `SPY`.
> * __Stress tests and the five gates:__ We ran strategies on 200 synthetic markets and tested their scores against five gates. Few strategies passed the failure-rate gate. Even `SPY` ended below the starting wealth on about a quarter of the markets.

A backtest and a stress test use past and synthetic prices. Next time, we'll trade a stress-tested strategy in a paper account at live market prices.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___